# Exercise D — Model Reduction with POD

**Companion to Lecture D** (Model Reduction with POD) of the lecture notes *Essential Concepts in FEM
and Physics-Based ML*. Needs Lecture 5 and Exercise 5; uses the least squares of Lecture A.

You **write the code** yourself: a POD routine, a reduced model by Galerkin projection, and a reduced
model by regression. You test each on the hand examples of the lecture and then on the bar of
Exercise 5, and you look for the point where the idea stops working.

**What the small examples are for.** A bar with 20 unknowns needs no reduction: the full model is
solved faster than it is reduced. The examples are small so that every step can be checked and the
*structure* of the method can be seen. Keep that in mind when you read the timings and sizes below.

Only `numpy` and `matplotlib`. Fill in every `...`; each part ends with a ✅ check cell.

---
*Author: Ramses Sala · Copyright © 2025 Ramses Sala · Part of the course* Essential Concepts in FEM and Physics-Based ML. *Created with assistance of Claude Code for text, image and code refinement and generation.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)


The next cell contains the solver pieces of Exercise 5 (assembly of a chain, the bar, natural
frequencies, the central-difference stepper without damping). If you have done Exercise 5, you may
paste your own versions instead.

In [ ]:
# --- the tools of Exercise 5 (replace them by your own versions if you like) ---
def assemble_chain(k_list, m_list):
    n = len(k_list)
    K_full = np.zeros((n + 1, n + 1))
    for e, k in enumerate(k_list):
        K_full[e:e+2, e:e+2] += k*np.array([[1.0, -1.0], [-1.0, 1.0]])
    return K_full[1:, 1:], np.asarray(m_list, dtype=float)

def bar_chain(ne, EA=1.0, rhoA=1.0, L=1.0):
    h = L/ne
    return assemble_chain([EA/h]*ne, [rhoA*h]*(ne - 1) + [rhoA*h/2])

def natural_frequencies(K, m):
    s = 1/np.sqrt(m)
    lam, Y = np.linalg.eigh(s[:, None]*K*s[None, :])
    return np.sqrt(lam), s[:, None]*Y

def central_difference(K, m, f_func, u0, v0, dt, nsteps):
    a0 = (f_func(0.0) - K @ u0)/m
    u_prev = u0 - dt*v0 + 0.5*dt**2*a0
    u = u0.astype(float).copy()
    U = np.zeros((len(m), nsteps + 1)); U[:, 0] = u
    for n in range(nsteps):
        u_new = 2*u - u_prev + dt**2*(f_func(n*dt) - K @ u)/m
        u_prev, u = u, u_new
        U[:, n + 1] = u
    return np.arange(nsteps + 1)*dt, U

def rel_err(A, B):
    return np.linalg.norm(A - B)/np.linalg.norm(A)


---
## Part 1 — A POD routine, checked by hand

Write `pod(S, energy)`: it takes a snapshot matrix (one solution per column) and returns

* `Phi` — the POD shapes as columns, most important first,
* `sigma` — all singular values,
* `r` — the smallest number of shapes whose share
  $(\sigma_1^2+\dots+\sigma_r^2)/(\sigma_1^2+\sigma_2^2+\dots)$ reaches `energy`.

Use `np.linalg.svd(S, full_matrices=False)`. `np.cumsum` and `np.searchsorted` (or `np.argmax` on a
boolean array) are useful for `r`.

Then test it on the two hand examples of the lecture, both for the chain of three unit springs:

1. end loads $F = 1, 2, 3$ — the rank-one table (Section D.2): $\sigma_1 = 14$;
2. a unit load at the end and one at the middle node (Section D.3): $\sigma = 4.77,\ 0.47$, first share $99.0\,\%$.

Build both tables by **solving** $\mathbf K\mathbf u = \mathbf f$, not by typing the numbers.

In [ ]:
def pod(S, energy=0.999):
    Phi, sigma, Vt = ...                      # TODO: the SVD
    share = ...                               # TODO: cumulative share of sigma^2
    r = ...                                   # TODO: smallest r with share >= energy
    return Phi, sigma, r

K3, _ = assemble_chain([1.0, 1.0, 1.0], [1, 1, 1])
S_rank1 = ...                                 # TODO: columns = solutions for end loads 1, 2, 3
S_two = ...                                   # TODO: columns = end load, middle load
Phi_a, sig_a, r_a = pod(S_rank1)
Phi_b, sig_b, r_b = pod(S_two, energy=0.99)
print("rank-one table:\n", S_rank1, "\nsigma =", sig_a, " r =", r_a)
print("two loads:\n", S_two, "\nsigma =", sig_b, " share of shape 1 =", sig_b[0]**2/np.sum(sig_b**2), " r =", r_b)
print("first shape:", Phi_b[:, 0])

In [ ]:
# ✅ check — Part 1
assert np.allclose(S_rank1, [[1, 2, 3], [2, 4, 6], [3, 6, 9]]) and np.allclose(S_two, [[1, 1], [2, 2], [3, 2]])
assert np.allclose(sig_a, [14, 0, 0], atol=1e-10) and r_a == 1, "One shape, sigma = 14 (Section D.2)."
assert np.allclose(sig_b, [4.7729, 0.4685], atol=1e-4) and r_b == 1, "Compare with the hand computation of Section D.3."
assert np.allclose(np.abs(Phi_b[:, 0]), [0.2945, 0.5889, 0.7527], atol=1e-4)
assert np.isclose(np.sum(sig_b**2), np.sum(S_two**2)), "The sigma^2 share out the sum of squares of all entries."
print("OK. (The sign of a shape is arbitrary: -phi with -v is the same rank-one piece.)")

---
## Part 2 — POD shapes and vibration modes

The run of Exercise 5, Part 6: a bar of 20 elements, force $F = 1$ switched on at the free end, to
$t = 8$ with $\Delta t = 0.9\,h$. Its snapshot matrix `U` has 20 rows and 178 columns.

1. Compute the POD of `U`. Print the first four singular values and their shares.
2. Compute the vibration modes and scale each to unit length. For the first four pairs
   (POD shape $k$, mode $k$) print $|\cos|$ of the angle between them, i.e. the absolute dot product.
3. Compute the relative error of the response reconstructed
   * from the first $r$ POD shapes: $\mathbf U_r = \boldsymbol\Phi_r\,(\boldsymbol\Phi_r^T\mathbf U)$,
   * from the first $r$ modes, as in Exercise 5: $\mathbf U_r = \boldsymbol\Psi_r\,(\boldsymbol\Psi_r^T\mathbf M\,\mathbf U)$ with mass-normalized modes $\boldsymbol\Psi$,

   for $r = 1, 2, 3, 6$. (`rel_err(A, B)` is given above.)
4. Plot the first three POD shapes and modes.

In [ ]:
ne = 20; h = 1/ne
K, m = bar_chain(ne)
F = np.zeros(ne); F[-1] = 1.0
dt = 0.9*h
t, U = central_difference(K, m, lambda t: F, np.zeros(ne), np.zeros(ne), dt, int(8/dt))

Phi, sigma, r999 = ...                        # TODO: POD of U
shares = ...                                  # TODO: sigma_i^2 / sum
omega, Psi = natural_frequencies(K, m)        # mass-normalized modes
Psi_unit = ...                                # TODO: the same modes, each scaled to unit length
cosines = ...                                 # TODO: |phi_k . psi_k| for k = 1..4

err_pod, err_modes = {}, {}
for r in (1, 2, 3, 6):
    err_pod[r] = ...                          # TODO
    err_modes[r] = ...                        # TODO
print("sigma:", sigma[:4], " shares:", shares[:4], " r for 99.9 %:", r999)
print("|cos| POD shape k / mode k:", cosines)
print("error with r POD shapes:", {r: round(e, 4) for r, e in err_pod.items()})
print("error with r modes:     ", {r: round(e, 4) for r, e in err_modes.items()})

x = np.linspace(h, 1, ne)
fig, ax = plt.subplots(1, 3, figsize=(11, 2.8))
for k in range(3):
    sgn = np.sign(Phi[:, k] @ Psi_unit[:, k])
    ax[k].plot(x, sgn*Phi[:, k], 'b-', label='POD shape'); ax[k].plot(x, Psi_unit[:, k], 'k--', label='mode')
    ax[k].set_title(f'shape {k+1}'); ax[k].set_xlabel('x')
ax[0].legend(); plt.show()

In [ ]:
# ✅ check — Part 2
assert U.shape == (20, 178)
assert np.allclose(sigma[:4], [43.39, 3.685, 1.220, 0.601], atol=5e-3), "Compare with Section D.4."
assert r999 == 3 and shares[0] > 0.99
assert np.allclose(cosines, [0.996, 0.968, 0.929, 0.897], atol=2e-3)
assert all(err_pod[r] < err_modes[r] for r in (1, 2, 3, 6)), "No r shapes can beat the first r POD shapes on these snapshots."
assert abs(err_pod[3] - 0.0184) < 1e-3 and abs(err_modes[3] - 0.0311) < 1e-3
x_line = x/np.linalg.norm(x)
print(f"OK. First POD shape against the straight line (static deflection): |cos| = {abs(Phi[:, 0] @ x_line):.4f}")

---
## Part 3 — A reduced model by Galerkin projection

**3a — by hand first.** The three-spring chain with the single shape $\boldsymbol\phi = [1\ 2\ 3]^T$.
Write `galerkin_static(Phi, K, f)`: form $\mathbf K_r = \boldsymbol\Phi^T\mathbf K\boldsymbol\Phi$ and
$\mathbf f_r = \boldsymbol\Phi^T\mathbf f$, solve for the amounts $\mathbf a$, return $\boldsymbol\Phi\mathbf a$.
Check the two cases of Section D.5: an end load (exact) and a unit load on every node ($[2\ 4\ 6]$
against the exact $[3\ 5\ 6]$).

**3b — in time.** Write `reduced_run(Phi_r, K, m, f, dt, nsteps)` for a constant load vector `f`,
starting at rest: central differences on
$\mathbf M_r\ddot{\mathbf a} + \mathbf K_r\mathbf a = \mathbf f_r$. $\mathbf M_r = \boldsymbol\Phi_r^T\mathbf M\boldsymbol\Phi_r$
is small but **not diagonal**, so each step solves a tiny $r\times r$ system (invert $\mathbf M_r$ once).
Return the full-size response $\boldsymbol\Phi_r\mathbf A$ and the largest stable time step
$2/\omega_{\max}$ of the reduced model ($\omega^2$ = eigenvalues of $\mathbf M_r^{-1}\mathbf K_r$).

Then, with the POD shapes of Part 2:

1. errors of the reduced model against `U` for $r = 1, 2, 3, 5, 8$, and the stable step of each;
2. rerun $r = 3$ with $\Delta t = 0.2$ — four times the step the full model can take;
3. an **unseen load**: the same force at the middle node instead of the end. Compute the full
   response and the error of the $r = 3$ model built from the *end-load* shapes.

In [ ]:
def galerkin_static(Phi, K, f):
    Kr = ...                                  # TODO
    fr = ...                                  # TODO
    a = np.linalg.solve(Kr, fr)
    return Phi @ a

phi = np.array([[1.0], [2.0], [3.0]])
u_end = galerkin_static(phi, K3, np.array([0, 0, 1.0]))
u_all = galerkin_static(phi, K3, np.array([1.0, 1.0, 1.0]))
print("end load:", u_end, "  load on every node:", u_all, "  exact:", np.linalg.solve(K3, [1, 1, 1]))

def reduced_run(Phi_r, K, m, f, dt, nsteps):
    Kr = ...                                  # TODO
    Mr = ...                                  # TODO: m is a vector (diagonal mass matrix)
    fr = ...                                  # TODO
    Mr_inv = np.linalg.inv(Mr)
    w_max = np.sqrt(np.max(np.linalg.eigvals(Mr_inv @ Kr).real))
    r = Phi_r.shape[1]
    a = np.zeros(r)
    a_prev = 0.5*dt**2*(Mr_inv @ fr)          # start at rest: a^{-1} = a^0 + dt^2/2 * acceleration
    A = np.zeros((r, nsteps + 1))
    for n in range(nsteps):
        a_new = ...                           # TODO: central difference for the reduced system
        a_prev, a = a, a_new
        A[:, n + 1] = a
    return Phi_r @ A, 2/w_max

nsteps = int(8/dt)
rom_err, rom_dt = {}, {}
for r in (1, 2, 3, 5, 8):
    U_rom, dt_lim = reduced_run(Phi[:, :r], K, m, F, dt, nsteps)
    rom_err[r] = rel_err(U, U_rom); rom_dt[r] = dt_lim
print("error of the reduced model:", {r: round(e, 4) for r, e in rom_err.items()})
print("largest stable step:       ", {r: round(d, 3) for r, d in rom_dt.items()}, "  full model:", round(2/omega[-1], 4))

U_big, _ = reduced_run(Phi[:, :3], K, m, F, 0.2, 40)
print("r = 3 with dt = 0.2: 40 steps instead of", nsteps, "  tip maximum", round(U_big[-1].max(), 3),
      "  (full model:", round(U[-1].max(), 3), ")")

F_mid = np.zeros(ne); F_mid[ne//2 - 1] = 1.0
_, U_mid = central_difference(K, m, lambda t: F_mid, np.zeros(ne), np.zeros(ne), dt, nsteps)
U_mid_rom, _ = reduced_run(Phi[:, :3], K, m, F_mid, dt, nsteps)
err_unseen = rel_err(U_mid, U_mid_rom)
err_unseen_proj = rel_err(U_mid, Phi[:, :3] @ (Phi[:, :3].T @ U_mid))
print("unseen load, r = 3: reduced model error", round(err_unseen, 4), "  best possible with these shapes", round(err_unseen_proj, 4))

plt.plot(t, U[-1], color='0.6', lw=3, label='full')
for r, st in ((1, 'r--'), (3, 'b-')):
    plt.plot(t, reduced_run(Phi[:, :r], K, m, F, dt, nsteps)[0][-1], st, label=f'r = {r}')
plt.xlabel('t'); plt.ylabel('tip displacement'); plt.legend(); plt.show()

In [ ]:
# ✅ check — Part 3
assert np.allclose(u_end, [1, 2, 3]), "The end-load solution lives in the family: Galerkin finds it exactly."
assert np.allclose(u_all, [2, 4, 6]), "3a = 6, so u = [2, 4, 6] (exact: [3, 5, 6])."
assert abs(rom_err[1] - 0.216) < 5e-3 and abs(rom_err[3] - 0.053) < 5e-3 and rom_err[8] < 0.02
assert all(rom_err[r] > err_pod[r] for r in (1, 2, 3)), "The projection error of Part 2 is a floor, not a forecast."
assert abs(rom_dt[3] - 0.235) < 5e-3 and rom_dt[1] > 1.0
assert np.isfinite(U_big).all() and 1.8 < U_big[-1].max() < 2.1, "dt = 0.2 is stable for r = 3."
print("OK. Fewer unknowns, and a time step limit several times larger: the stiffest modes are gone.")

---
## Part 4 — A reduced model by regression (no $\mathbf K$ needed)

A bar of 20 elements, fixed at $x=0$, under a uniform distributed load $n_0 = 1$ (nodal forces $h$ on
interior nodes, $h/2$ on the end node). The elements with midpoints in $0.4 < x < 0.6$ are weakened:
their stiffness is $\mu\,EA$. The parameter is $\mu$.

1. Write `soft_bar(mu)` returning the nodal displacements and the stiffness matrix (use `assemble_chain`).
2. Snapshots for $\mu = 0.2, 0.4, 0.6, 0.8, 1.0$. POD: how many singular values are not zero? Why?
3. POD amounts of the snapshots: $\mathbf a(\mu_k) = \boldsymbol\Phi_r^T\mathbf u^{(k)}$ with $r = 2$.
   Fit them by least squares (`np.linalg.lstsq`, the normal equations of Lecture A) with three
   different families:

   * linear in $\mu$: features $[1,\ \mu]$,
   * quadratic in $\mu$: features $[1,\ \mu,\ \mu^2]$,
   * linear in $1/\mu$: features $[1,\ 1/\mu]$.
4. Test all three at 41 values of $\mu$ between 0.2 and 1: largest relative error against the full
   solution. Add the Galerkin reduced model `galerkin_static(Phi_r, K(mu), f)` as a fourth row.

In [ ]:
ne = 20; h = 1/ne
x_mid = (np.arange(ne) + 0.5)*h
f_uniform = np.full(ne, h); f_uniform[-1] = h/2

def soft_bar(mu):
    k_list = ...                              # TODO: EA/h, times mu in the weakened zone
    Kmu, _ = assemble_chain(k_list, np.ones(ne))
    return np.linalg.solve(Kmu, f_uniform), Kmu

mu_train = np.array([0.2, 0.4, 0.6, 0.8, 1.0])
S_mu = ...                                    # TODO: snapshot matrix, one column per mu
Phi_mu, sig_mu, _ = pod(S_mu)
print("singular values:", sig_mu)
Phi2 = Phi_mu[:, :2]
A_train = ...                                 # TODO: amounts, shape (2, 5)

families = {
    "linear in mu":    lambda mu: np.column_stack([np.ones_like(mu), mu]),
    "quadratic in mu": lambda mu: np.column_stack([np.ones_like(mu), mu, mu**2]),
    "linear in 1/mu":  lambda mu: np.column_stack([np.ones_like(mu), 1/mu]),
}
mu_test = np.linspace(0.2, 1.0, 41)
U_test = np.column_stack([soft_bar(mu)[0] for mu in mu_test])
worst = {}
for name, feat in families.items():
    C = ...                                   # TODO: least squares, features(mu_train) @ C ~ A_train.T
    U_pred = ...                              # TODO: predicted full solutions at mu_test, shape (20, 41)
    worst[name] = max(rel_err(U_test[:, j], U_pred[:, j]) for j in range(len(mu_test)))
worst["Galerkin"] = max(rel_err(U_test[:, j], galerkin_static(Phi2, soft_bar(mu)[1], f_uniform))
                        for j, mu in enumerate(mu_test))
for name, e in worst.items():
    print(f"{name:18s} largest error {e:.2e}")

In [ ]:
# ✅ check — Part 4
assert sig_mu[1] > 0.1 and sig_mu[2] < 1e-10, "Exactly two shapes: u(mu) = u_a + (1/mu) u_b."
assert abs(worst["linear in mu"] - 0.12) < 0.01 and abs(worst["quadratic in mu"] - 0.061) < 0.005
assert worst["linear in 1/mu"] < 1e-10 and worst["Galerkin"] < 1e-10
print("OK. Same data, same least squares: the family decides. Galerkin did not need to be told about 1/mu.")

---
## Part 5 — Where POD fails: things that move

**5a — by hand.** Build the two $3\times3$ tables of Section D.7 — a pulse that sits on node 1, 2, 3 in
turn, and a front that has reached 1, then 2, then 3 nodes — and compute their singular values.

**5b — the wave front of Exercise 5.** In the first moments after the sudden load ($t < 1$) a front
travels along the bar. Write `shapes_needed(ne)` that, for a bar of `ne` elements,

* runs the sudden-load problem to $t = 8$ with $\Delta t = 0.9\,h$,
* forms the **velocity** snapshots $(\mathbf u^{n+1} - \mathbf u^n)/\Delta t$ and keeps those with $t < 1$,
* returns the number of shapes needed for 99.9 % of (i) the displacement snapshots of the whole run and
  (ii) the velocity snapshots of the front.

Run it for 20, 50 and 100 elements. Plot $\sigma_i/\sigma_1$ of both tables for 20 elements on a
logarithmic axis.

In [ ]:
S_pulse = ...                                 # TODO
S_front = ...                                 # TODO (columns: front has reached 1, 2, 3 nodes)
sig_pulse = np.linalg.svd(S_pulse, compute_uv=False)
sig_front = np.linalg.svd(S_front, compute_uv=False)
print("pulse:", sig_pulse, "  front:", sig_front, " squares:", sig_front**2,
      " share of the first shape:", sig_front[0]**2/np.sum(sig_front**2))

def front_tables(ne):
    h = 1/ne; dt = 0.9*h
    K, m = bar_chain(ne)
    F = np.zeros(ne); F[-1] = 1.0
    _, U = central_difference(K, m, lambda t: F, np.zeros(ne), np.zeros(ne), dt, int(8/dt))
    V = ...                                   # TODO: velocities between consecutive snapshots
    V_front = ...                             # TODO: keep the columns with t < 1
    return U, V_front

def shapes_needed(ne, energy=0.999):
    U, V_front = front_tables(ne)
    return pod(U, energy)[2], pod(V_front, energy)[2]

needed = {ne: shapes_needed(ne) for ne in (20, 50, 100)}
print("shapes for 99.9 %  (displacements, velocity front):", needed)

U20, V20 = front_tables(20)
for S_, lab in ((U20, 'displacements, whole run'), (V20, 'velocities, travelling front')):
    s_ = np.linalg.svd(S_, compute_uv=False)
    plt.semilogy(np.arange(1, len(s_) + 1), s_/s_[0], 'o-', label=lab)
plt.ylim(1e-4, 2); plt.xlabel('i'); plt.ylabel('sigma_i / sigma_1'); plt.legend(); plt.show()

In [ ]:
# ✅ check — Part 5
assert np.allclose(sig_pulse, 1), "A travelling pulse: all shapes equally important, nothing to drop."
assert np.allclose(sig_front**2, [5.049, 0.643, 0.308], atol=1e-3)
assert needed == {20: (3, 16), 50: (3, 37), 100: (3, 68)}, "Compare with the table of Section D.7."
print("OK. Displacements: 3 shapes on every mesh. The moving front: the finer the mesh, the more shapes.")

---
## Part 6 (optional) — Bridge to 2D: a plate with a moving inclusion

*For readers who know the triangles of Lecture 4 and the heated plate of Lecture C. Nothing in the
other parts depends on this one.*

A square plate, heated uniformly ($s = 1$), edges held at $T = 0$, with an inclusion of higher
conductivity,
$\lambda(x, y) = 1 + d\,\exp\bigl(-((x-x_c)^2 + (y-y_c)^2)/w_z^2\bigr)$, $d = 1$, $w_z = 0.15$ — the
inclusion of Section C.9, here in a uniformly heated plate. The parameter is the **position** $(x_c, y_c)$ of the inclusion. A small solver on
linear triangles is given below; a snapshot is the vector of all nodal temperatures, and POD does
not care that the nodes now sit in a plane.

1. Compute 25 snapshots: $x_c, y_c \in \{0.25, 0.375, 0.5, 0.625, 0.75\}$. (Takes a few seconds.)
2. POD of the snapshot matrix: share of the first shape? Compare that shape with the temperature
   field of the plate **without** inclusion.
3. Subtract the field without inclusion from every snapshot and take the POD of the *deviations*.
   Shares of the first 3 and the first 8 shapes?
4. An unseen position, $(0.6, 0.4)$: how well do $r = 3$ and $r = 8$ deviation shapes reproduce what
   this inclusion changes?
5. Draw the first three deviation shapes with `plt.tricontourf(P[:, 0], P[:, 1], T, shape)`.

In [ ]:
# --- a small 2D heat solver on linear triangles (given) ---
def square_mesh(n):
    '''Unit square, n x n squares, each cut into two triangles.'''
    xs = np.linspace(0, 1, n + 1)
    X, Y = np.meshgrid(xs, xs)
    P = np.column_stack([X.ravel(), Y.ravel()])
    T = []
    for j in range(n):
        for i in range(n):
            a = j*(n + 1) + i; b = a + 1; c = a + n + 2; d = a + n + 1
            T += [[a, b, c], [a, c, d]]
    return P, np.array(T)

def solve_plate(P, T, conductivity, s=1.0):
    '''-div(lambda grad T) = s, T = 0 on all edges. Linear triangles, as in Lecture 4:
    the element matrix is lambda * A * B^T B with the constant gradient matrix B.'''
    N = len(P); K = np.zeros((N, N)); f = np.zeros(N)
    for t in T:
        xy = P[t]
        twoA = np.linalg.det(np.column_stack([np.ones(3), xy]))
        B = np.array([[xy[1, 1] - xy[2, 1], xy[2, 1] - xy[0, 1], xy[0, 1] - xy[1, 1]],
                      [xy[2, 0] - xy[1, 0], xy[0, 0] - xy[2, 0], xy[1, 0] - xy[0, 0]]])/twoA
        lam = conductivity(*xy.mean(axis=0))
        K[np.ix_(t, t)] += lam*(twoA/2)*B.T @ B
        f[t] += s*(twoA/2)/3
    edge = (P[:, 0] < 1e-12) | (P[:, 0] > 1 - 1e-12) | (P[:, 1] < 1e-12) | (P[:, 1] > 1 - 1e-12)
    free = ~edge
    u = np.zeros(N)
    u[free] = np.linalg.solve(K[np.ix_(free, free)], f[free])
    return u

def inclusion(xc, yc, d=1.0, wz=0.15):
    return lambda x, y: 1 + d*np.exp(-((x - xc)**2 + (y - yc)**2)/wz**2)


In [ ]:
P, T = square_mesh(20)
centres = [(xc, yc) for xc in np.linspace(0.25, 0.75, 5) for yc in np.linspace(0.25, 0.75, 5)]
S_plate = ...                                 # TODO: 25 snapshots as columns
T0 = solve_plate(P, T, lambda x, y: 1.0)      # the plate without inclusion

Phi_p, sig_p, _ = pod(S_plate)
share_p = sig_p**2/np.sum(sig_p**2)
cos_T0 = ...                                  # TODO: |cos| between the first shape and T0

D = ...                                       # TODO: deviations from T0
Phi_d, sig_d, _ = pod(D)
cum_d = np.cumsum(sig_d**2)/np.sum(sig_d**2)

dev_new = solve_plate(P, T, inclusion(0.6, 0.4)) - T0
err_new = {r: rel_err(dev_new, Phi_d[:, :r] @ (Phi_d[:, :r].T @ dev_new)) for r in (3, 8)}
print(f"full fields: first shape carries {share_p[0]:.4%}; |cos| with the plate without inclusion = {cos_T0:.5f}")
print(f"deviations: 3 shapes {cum_d[2]:.1%}, 8 shapes {cum_d[7]:.1%}")
print("unseen position, error in the deviation:", {r: round(e, 3) for r, e in err_new.items()})

fig, ax = plt.subplots(1, 3, figsize=(10, 3))
for k in range(3):
    ax[k].tricontourf(P[:, 0], P[:, 1], T, Phi_d[:, k], 20); ax[k].set_aspect('equal'); ax[k].set_title(f'deviation shape {k+1}')
plt.show()

In [ ]:
# ✅ check — Part 6
assert S_plate.shape == (441, 25)
assert share_p[0] > 0.999 and cos_T0 > 0.999, "The first shape is, practically, the plate without inclusion."
assert abs(cum_d[2] - 0.936) < 0.01 and abs(cum_d[7] - 0.992) < 0.005
assert err_new[8] < err_new[3] < 0.5
print("OK. The common field compresses to one shape; what the moving inclusion changes does not.")

---
## Closing reflections

1. In Part 2 the first POD shape was almost the straight line $u = x$. Why? What would the first POD
   shape be if you subtracted the static solution from every snapshot first?
2. In Part 3 the one-shape model oscillates with $\omega = 1.646$, the true lowest frequency of the
   mesh is $1.570$. Could a one-shape model ever give a frequency *below* $1.570$?
3. In Part 4, Galerkin was exact without being told about $1/\mu$, and so was one of the regressions.
   What did each of the two need to know about the problem? Which would you use if the snapshots
   came from a program you cannot open?
4. Part 5: the number of shapes for the front grew with the mesh. Suggest, in words, a different way
   of describing the snapshots of a travelling front with *few* numbers. (You are not asked to
   implement it.)
5. Every reduced model in this notebook was slower to *build* than the full model was to *solve*.
   Name the two properties a problem must have before the effort can pay.